# Walkthrough: from ticks to features

This notebook runs the whole pipeline on **synthetic data**: three fictional instruments and a
fictional benchmark index, with dates in 2030 so nothing can be mistaken for real history. The
same code runs on real tick data once it is mapped to the normalised trade table described in
`nordic_equity_features/schema.py`.

1. Generate a market
2. Look at the trade records
3. Clean the trades and build one-minute bars
4. Compute every feature family
5. Corporate actions and news, as the features see them

In [ ]:
import pandas as pd

import nordic_equity_features as nef
from nordic_equity_features.bars import build_bars, clean_trades, lit_trades

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#eeeeee"


def style(fig, title, height=520):
    fig.update_layout(
        title=dict(text=title, font=dict(size=18, color=INK)),
        height=height, margin=dict(l=60, r=20, t=70, b=40),
        plot_bgcolor="white", paper_bgcolor="white", font=dict(color=MUTED),
        hovermode="x unified", legend=dict(orientation="h", y=1.06, x=0),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zerolinecolor="#d0d0d0")
    return fig

## 1. Generate a market

Ten trading days. One instrument (`SKOG`) carries broker codes, one has a 2-for-1 split
(`FJORD`) and one pays a dividend announced in the middle of a session (`NORDA`).

In [ ]:
data = nef.generate(n_days=10, seed=7)
data.instruments

In [ ]:
data.trades.groupby("symbol").size().rename("trades").to_frame()

## 2. The trade records

Each print carries a **tradecode** bitfield (lit / off-book / dark / late / correction) and an
**MMT** string whose second character is the trading mode: `2` continuous, `O` opening auction,
`K` closing auction, `5` negotiated. The features rely on both.

In [ ]:
norda = data.trades.loc[data.trades["symbol"] == "NORDA"]
norda.head(8)

In [ ]:
from nordic_equity_features.schema import trading_mode
from nordic_equity_features.venue import classify_venue

pd.crosstab(classify_venue(norda), trading_mode(norda["mmt"]), margins=True)

## 3. Clean and build bars

Cleaning drops cancelled trades and correction / trade-break prints. Price bars use lit trades
only. A bar labelled `T` holds the trades in `[T − 1 min, T)`, so its values are complete at `T`.
Minutes without trades keep the last close and have zero volume.

In [ ]:
clean = clean_trades(norda)
lit = lit_trades(norda)
bars = build_bars(lit, data.days)
print(f"{len(norda):,} prints -> {len(clean):,} clean -> {len(lit):,} lit -> {len(bars):,} bars")
bars.head()

## 4. Every feature family

`compute_features` returns one row per bar and one column per feature. See
`docs/feature_catalogue.md` for every definition and its lookahead rule.

In [ ]:
features = {s: nef.compute_features(data, s) for s in data.instruments.index}
f = features["NORDA"]
print(f.shape)
f.iloc[200:205, :12]

In [ ]:
one_day = f.loc[str(data.days[3].date())]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("Close (lit)", "Volume imbalance (buyer − seller initiated)",
                                    "Realised volatility, 30 bars"))
fig.add_trace(go.Scatter(x=one_day.index, y=one_day["close"], name="close",
                         line=dict(color=BLUE, width=2)), row=1, col=1)
fig.add_trace(go.Bar(x=one_day.index, y=one_day["volume_imbalance"], name="volume imbalance",
                     marker_color=ORANGE), row=2, col=1)
fig.add_trace(go.Scatter(x=one_day.index, y=one_day["realised_vol"], name="realised vol",
                         line=dict(color=AQUA, width=2)), row=3, col=1)
style(fig, "Realised volatility peaks just after the open", height=700).show()

Venue mix per day: the four-way split of volume into lit, dark, off-book and unclassified prints.

In [ ]:
day = f.index.tz_convert("Europe/Stockholm").date
venue_mix = f.groupby(day)[["volume_lit", "volume_dark", "volume_offbook", "volume_residual"]].sum()
(venue_mix.div(venue_mix.sum(axis=1), axis=0) * 100).round(1)

Beta against the benchmark, estimated over a rolling 60-minute window. One-minute betas are
noisy and biased toward zero (bid-ask bounce dilutes the co-movement), which is why the
LASSO variant shrinks further. The dip at the open on 11 January is NORDA's dividend ex-day:
the price drops by the dividend at the open, and the 60-bar window carries that unadjusted
return for an hour.

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("OLS beta vs LASSO beta, 60 bars", "Correlation, 60 bars"))
fig.add_trace(go.Scatter(x=f.index, y=f["beta_ols"], name="beta (OLS)", line=dict(color=BLUE, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=f.index, y=f["beta_lasso"], name="beta (LASSO)", line=dict(color=ORANGE, width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=f.index, y=f["corr"], name="correlation", line=dict(color=AQUA, width=2)), row=2, col=1)
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"]), dict(bounds=[17.5, 9], pattern="hour")])
style(fig, "LASSO pulls the noisy one-minute beta toward zero", height=600).show()

## 5. Corporate actions and news, as the features see them

The dividend below was announced at 12:15. Bars up to and including 12:15 cannot see it; from
12:16 the countdown to the ex-date appears.

In [ ]:
data.dividends

In [ ]:
announced = data.dividends.loc[0, "announced"]
cols = ["dividend_ex_date", "days_to_ex_dividend", "dividend_declaration_flag", "dividend_amount", "dividend_pct"]
f.loc[announced - pd.Timedelta("2min"):announced + pd.Timedelta("2min"), cols]

On the split's ex-day the raw log return of the first bar is about −0.69: the price halves
because each share became two. The split-adjusted return removes that.

In [ ]:
fj = features["FJORD"]
ex = data.splits.loc[0, "ex_date"]
fj.loc[str(ex.date())].head(3)[["close", "log_return", "split_adjusted_return", "stock_split_xday_lookback"]]

In [ ]:
from nordic_equity_features.corporate_actions import adjusted_ohlcv

as_of_before = pd.Timestamp(f"{(ex - pd.offsets.BDay(1)).date()} 17:30", tz="Europe/Stockholm")
as_of_after = pd.Timestamp(f"{ex.date()} 17:30", tz="Europe/Stockholm")
pd.DataFrame({
    "history seen the day before": adjusted_ohlcv(fj, data.splits, as_of_before)["ac"].iloc[[0, -1]].to_numpy(),
    "history seen on the ex-day": adjusted_ohlcv(fj, data.splits, as_of_after)["ac"].iloc[[0, 1]].to_numpy(),
}, index=["first bar", "later bar"])

News is attached to the first bar after it is published. Items released before the open land
in the 09:01 bar.

In [ ]:
corp = data.news.loc[(data.news["scope"] == "corporate") & (data.news["symbol"] == "NORDA")].head(5)
corp.assign(first_bar_that_sees_it=[f.index[f.index > t][0] for t in corp["timestamp"]])